# DEE — Tests rigurosos de fase de vidrio (post-EA)

**Pregunta:** confirmar o refutar definitivamente la naturaleza vítrea del sustrato DEE en su fase fundamental, dado que el análisis Edwards-Anderson previo dió firma estática débil (q_κ ≈ 0) pero firma dinámica positiva (β = 0.56 ± 0.02 en stretched exp).

**Tres tests independientes con criterios pre-registrados:**

| Test | Mide | Criterio de aceptación |
|---|---|---|
| **1. Aging** | Rotura de TTI: τ_relax(t_w) creciente | τ(t_w=25k) / τ(t_w=1k) > 2.0 |
| **2. χ_4(t)** | Cooperatividad dinámica | peak χ_4 > 5 con N=500 |
| **3. Sweep N** | Escala finita de ξ_4 | χ_4_max(2N)/χ_4_max(N) > 1.5 |

**Combinación de veredictos:** los criterios son *pre-registrados* (definidos antes de mirar resultados, no a posteriori). Las 8 combinaciones binarias mapean a interpretaciones específicas en la celda final.

**Protocolo:**
1. Annealing pre-T=0 idéntico al SIM 18 / experimento EA: T schedule = [43.4, 14.47, 4.34], pasos = [7500, 7500, 4500] (total 19500).
2. **Quench instantáneo a T=0** (sin annealing gradual), define t_w = 0.
3. MC en T=0 con muestreo denso (log + linear): pasos {10, 30, 100, 300, 1000, 1500, 2000, ..., 50000}.
4. 8 réplicas independientes (seeds 1101..1402) — menos réplicas que en EA pero cada una con sampling 17× más denso en la fase fría.

**Configuración:**
- N = 500 (compatible con experimento EA previo)
- Topología: T³, L = 1.0
- F = N·var(κ) + α·Σ(1−I)² (idéntico a SIM 18)
- 8 réplicas × (12 min pre + 31 min T=0) ≈ **6 horas estimadas**
- Reanudable: cada réplica se guarda al terminar.

**Test 3 (sweep N) es opcional** y desactivado por default. Activar `RUN_SIZE_SCALING = True` solo si Tests 1 ó 2 dan positivo. Costo adicional: ~14 horas.

---

In [ ]:
import os
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix, diags
from scipy.optimize import curve_fit
from scipy.stats import skew, kurtosis
import matplotlib.pyplot as plt
import time
import json

PLOT_DIR = 'plots_aging'
DATA_DIR = 'data_aging'
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

def save_plot(name):
    plt.savefig(os.path.join(PLOT_DIR, f'{name}.png'), dpi=120, bbox_inches='tight')
    print(f'  -> {name}.png')

# Configuración
THETA_D = 43.4
N_TARGET = 500
K = 20
L_T3 = 1.0
N_REPLICAS = 8

# Schedule pre-T=0 (idéntico a SIM 18 / EA)
T_SCHEDULE_PRE = [THETA_D, THETA_D/3, THETA_D/10]   # 43.4, 14.47, 4.34
N_STEPS_PRE = [7500, 7500, 4500]                    # total 19500

# Schedule de muestreo a T=0 (log + linear denso)
N_T0_STEPS = 50000  # cuántos pasos a T=0 después del quench
def build_t0_schedule(t_max):
    """Pasos donde se guarda snapshot durante T=0."""
    log_part = [10, 30, 100, 300, 600, 1000]
    # Linear denso en zona temprana, espaciado mayor después
    lin_early = list(range(1500, 5001, 500))      # 1500, 2000, ..., 5000
    lin_mid   = list(range(6000, 20001, 1000))    # 6000, 7000, ..., 20000
    lin_late  = list(range(22000, t_max+1, 2000)) # 22000, 24000, ..., t_max
    sched = sorted(set(log_part + lin_early + lin_mid + lin_late))
    return [s for s in sched if s <= t_max]

T0_SAMPLE_SCHEDULE = build_t0_schedule(N_T0_STEPS)

# Flag para Test 3 (caro)
RUN_SIZE_SCALING = False

print(f'N = {N_TARGET}, k = {K}, L = {L_T3}')
print(f'Réplicas = {N_REPLICAS}')
print(f'Schedule pre-T=0: {T_SCHEDULE_PRE}, pasos = {N_STEPS_PRE} (total {sum(N_STEPS_PRE)})')
print(f'Pasos a T=0 después del quench: {N_T0_STEPS}')
print(f'Snapshots durante T=0: {len(T0_SAMPLE_SCHEDULE)}')
print(f'  primeros: {T0_SAMPLE_SCHEDULE[:8]}')
print(f'  últimos: {T0_SAMPLE_SCHEDULE[-5:]}')
print(f'Test 3 (sweep N): {"ACTIVADO" if RUN_SIZE_SCALING else "desactivado (default)"}')

In [ ]:
# Funciones core (IDÉNTICAS a DEE_glass_test.ipynb)

def init_uniform_T3(N_target, seed=42, L=1.0):
    rng = np.random.default_rng(seed)
    return rng.uniform(0, L, size=(N_target, 3)), N_target

def neighbors_T3_fast(points, k=20, L=1.0):
    N = len(points)
    images = []
    image_to_orig = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:
                shift = np.array([dx*L, dy*L, dz*L])
                images.append(points + shift)
                image_to_orig.append(np.arange(N))
    images_all = np.concatenate(images, axis=0)
    image_to_orig_all = np.concatenate(image_to_orig)
    tree = cKDTree(images_all)

    nbrs = np.zeros((N, k), dtype=int)
    nbr_dists = np.zeros((N, k))
    nbr_pos = np.zeros((N, k, 3))
    for i in range(N):
        dists, idxs = tree.query(points[i], k=k+1)
        valid = ~((image_to_orig_all[idxs] == i) & (dists < 1e-12))
        v_idx = np.where(valid)[0][:k]
        nbrs[i] = image_to_orig_all[idxs[v_idx]]
        nbr_dists[i] = dists[v_idx]
        nbr_pos[i] = images_all[idxs[v_idx]]
    return nbrs, nbr_dists, nbr_pos

def laplacian_from_neighbors(nbrs, nbr_dists, alpha=2.0):
    N = len(nbrs)
    k = nbrs.shape[1]
    rows = np.repeat(np.arange(N), k)
    cols = nbrs.ravel()
    weights = (1.0 / nbr_dists**alpha).ravel()
    W = csr_matrix((weights, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    deg = np.array(W.sum(axis=1)).flatten()
    return diags(deg) - W, deg

def compute_F_vec(points, alpha_iso, alpha_kernel=2.0, k=K, L=L_T3):
    nbrs, nbr_dists, nbr_pos = neighbors_T3_fast(points, k=k, L=L)
    deg = np.sum(1.0 / nbr_dists**alpha_kernel, axis=1)
    kappa = deg / np.mean(deg)
    F_var = len(points) * np.var(kappa)

    if alpha_iso > 0:
        diffs = nbr_pos - points[:, None, :]
        M = np.einsum('ikj,ikm->ijm', diffs, diffs) / nbrs.shape[1]
        eigvals_all = np.linalg.eigvalsh(M)
        eigvals_all = np.maximum(eigvals_all, 1e-12)
        eigvals_top = eigvals_all[:, -3:] if eigvals_all.shape[1] > 3 else eigvals_all
        geom = np.exp(np.mean(np.log(eigvals_top), axis=1))
        arith = np.mean(eigvals_top, axis=1)
        iso = geom / arith
        F_iso = np.sum((1.0 - iso)**2)
        F_total = F_var + alpha_iso * F_iso
    else:
        iso = None
        F_total = F_var
        F_iso = 0
    return F_total, F_var, F_iso, kappa, iso

print('Funciones core listas')

In [ ]:
# Calibración de alpha_iso (mismo procedimiento que SIM 18 / EA)
pts_init_for_calib, _ = init_uniform_T3(N_TARGET, seed=42, L=L_T3)
_, F_var_r, F_iso_r, _, _ = compute_F_vec(pts_init_for_calib, alpha_iso=1.0)
ALPHA_ISO = F_var_r / max(F_iso_r, 1e-6)
print(f'α_iso calibrado = {ALPHA_ISO:.4f}')
print(f'(igualdad de contribuciones para configuración aleatoria)')

In [ ]:
# MC con annealing + sampling configurable
# Generaliza run_mc_replica del notebook EA para aceptar una lista
# explícita de pasos donde guardar snapshots de κ.

def gen_move_T3(rng, sigma):
    return rng.normal(0, sigma, size=3)

def move_T3(points, i, delta, L=1.0):
    new_pts = points.copy()
    new_pts[i] = (points[i] + delta) % L
    return new_pts

def run_mc_phase(pts_init, T, n_steps, alpha_iso, sigma_move=0.02,
                 seed=42, sample_at=None, F_init=None, verbose=True):
    """
    Una única fase MC a temperatura T.
    sample_at: lista/array de pasos (relativos al inicio de la fase) donde guardar
               snapshot completo de κ. Si None, sampleo cada 2000 pasos.
    F_init: F inicial (para evitar recomputarla). Si None, se calcula.
    Retorna dict con pts_final, F_curr, kappa_samples, sample_steps_actual.
    """
    rng = np.random.default_rng(seed)
    points = pts_init.copy()
    if F_init is None:
        F_curr, _, _, _, _ = compute_F_vec(points, alpha_iso)
    else:
        F_curr = F_init

    if sample_at is None:
        sample_at = list(range(0, n_steps, 2000))
    sample_at_set = set(int(s) for s in sample_at)

    kappa_samples = []
    sample_steps_actual = []
    F_at_samples = []
    n_acc = 0
    t_start = time.time()

    # Snapshot en paso 0 si está pedido
    if 0 in sample_at_set:
        _, _, _, k_, _ = compute_F_vec(points, max(alpha_iso, 1.0))
        kappa_samples.append(k_.copy())
        sample_steps_actual.append(0)
        F_at_samples.append(F_curr)

    for s in range(1, n_steps + 1):
        i = rng.integers(len(points))
        delta = gen_move_T3(rng, sigma_move)
        pts_new = move_T3(points, i, delta, L=L_T3)
        F_new, _, _, _, _ = compute_F_vec(pts_new, alpha_iso)
        dF = F_new - F_curr

        if T > 0:
            accept = (dF < 0) or (rng.random() < np.exp(-dF / T))
        else:
            accept = (dF < 0)

        if accept:
            points = pts_new
            F_curr = F_new
            n_acc += 1

        if s in sample_at_set:
            _, _, _, k_, _ = compute_F_vec(points, max(alpha_iso, 1.0))
            kappa_samples.append(k_.copy())
            sample_steps_actual.append(s)
            F_at_samples.append(F_curr)

    ar = n_acc / n_steps
    elapsed = time.time() - t_start
    if verbose:
        print(f'    T={T:6.2f}  acept={ar*100:5.1f}%  F={F_curr:.2f}  '
              f'snaps={len(kappa_samples)}  t={elapsed:.0f}s')

    return {
        'pts_final': points,
        'F_final': F_curr,
        'kappa_samples': np.array(kappa_samples),
        'sample_steps': np.array(sample_steps_actual),
        'F_at_samples': np.array(F_at_samples),
        'accept_rate': ar,
        'elapsed': elapsed
    }

print(f'run_mc_phase listo')
print(f'Schedule pre-T=0: {T_SCHEDULE_PRE}, pasos={N_STEPS_PRE}, total={sum(N_STEPS_PRE)}')
print(f'Schedule T=0: {N_T0_STEPS} pasos, {len(T0_SAMPLE_SCHEDULE)} snapshots')

In [ ]:
# CONFIGURACIÓN INICIAL COMÚN
# Misma config que EA: todas las réplicas parten del mismo pts_initial,
# diferentes seeds para la secuencia de movimientos MC.

INITIAL_SEED = 42
pts_initial_common, _ = init_uniform_T3(N_TARGET, seed=INITIAL_SEED, L=L_T3)
F_init_common, _, _, kappa_init, _ = compute_F_vec(pts_initial_common, ALPHA_ISO)
print(f'Configuración inicial común (seed={INITIAL_SEED}):')
print(f'  N = {len(pts_initial_common)}')
print(f'  F inicial = {F_init_common:.2f}')
print(f'  var(κ) inicial = {kappa_init.var():.4f}')

np.save(os.path.join(DATA_DIR, 'pts_initial.npy'), pts_initial_common)
print(f'Configuración inicial guardada en {DATA_DIR}/pts_initial.npy')

---
## EJECUCIÓN AUTÓNOMA — 8 réplicas con protocolo de aging

Cada réplica:
1. Annealing pre-T=0 (3 fases, 19500 pasos) — sampling cada 2000 pasos
2. **Quench instantáneo a T=0** (sin gradiente intermedio)
3. MC a T=0 con sampling denso log+linear (50000 pasos, ~50 snapshots)

Cada réplica se guarda en `data_aging/aging_XX_seedYYY.npz` al terminar. Si la celda se corta, al re-ejecutar saltea las réplicas que ya están guardadas.

In [ ]:
# Loop principal: 8 réplicas con protocolo aging
# Seeds nuevas, no se solapan con las del EA (101..602)

REPLICA_SEEDS_AGING = [1101, 1102, 1201, 1202, 1301, 1302, 1401, 1402]

all_aging = {}
t_global_start = time.time()

for rep_idx, rep_seed in enumerate(REPLICA_SEEDS_AGING):
    fname = os.path.join(DATA_DIR, f'aging_{rep_idx:02d}_seed{rep_seed}.npz')

    # Skip si ya existe
    if os.path.exists(fname):
        print(f'\n[{rep_idx+1}/{N_REPLICAS}] Réplica {rep_idx} (seed {rep_seed}) ya existe, cargando...')
        d = np.load(fname, allow_pickle=True)
        all_aging[rep_idx] = {
            'pts_final': d['pts_final'],
            'kappa_final': d['kappa_final'],
            'F_final': float(d['F_final']),
            'kappa_samples_pre': d['kappa_samples_pre'],
            'sample_steps_pre': d['sample_steps_pre'],
            'kappa_samples_T0': d['kappa_samples_T0'],
            'sample_steps_T0': d['sample_steps_T0'],
            'F_at_T0_samples': d['F_at_T0_samples'],
            'seed': rep_seed,
            'time': float(d['total_time'])
        }
        print(f'    cargada (F_final = {all_aging[rep_idx]["F_final"]:.2f})')
        continue

    print(f'\n{"="*70}')
    print(f'[{rep_idx+1}/{N_REPLICAS}] Réplica {rep_idx} (seed {rep_seed})')
    print(f'{"="*70}')
    elapsed_global = time.time() - t_global_start
    if rep_idx > 0:
        avg_per_replica = elapsed_global / rep_idx
        eta_min = avg_per_replica * (N_REPLICAS - rep_idx) / 60
        print(f'Tiempo transcurrido: {elapsed_global/60:.0f} min, ETA: {eta_min:.0f} min restantes')

    t_replica_start = time.time()

    # ── FASE 1: Annealing pre-T=0 ──────────────────────────
    pts_curr = pts_initial_common.copy()
    F_curr = F_init_common
    kappa_samples_pre_list = []
    sample_steps_pre_list = []
    offset = 0

    for phase_idx, (T_phase, n_steps) in enumerate(zip(T_SCHEDULE_PRE, N_STEPS_PRE)):
        print(f'\n  Fase pre-T=0 [{phase_idx+1}/3]: T = {T_phase:.2f}, {n_steps} pasos')
        sample_at_phase = list(range(0, n_steps, 2000))
        result = run_mc_phase(
            pts_curr, T_phase, n_steps, alpha_iso=ALPHA_ISO,
            sigma_move=0.02, seed=rep_seed + phase_idx,
            sample_at=sample_at_phase, F_init=F_curr
        )
        pts_curr = result['pts_final']
        F_curr = result['F_final']
        kappa_samples_pre_list.append(result['kappa_samples'])
        sample_steps_pre_list.append(result['sample_steps'] + offset)
        offset += n_steps

    kappa_samples_pre = np.concatenate(kappa_samples_pre_list, axis=0)
    sample_steps_pre = np.concatenate(sample_steps_pre_list)

    # ── FASE 2: Quench a T=0 + sampling denso ──────────────
    print(f'\n  ↓↓ QUENCH a T=0 ↓↓')
    print(f'  Fase T=0: {N_T0_STEPS} pasos, {len(T0_SAMPLE_SCHEDULE)} snapshots')
    result_T0 = run_mc_phase(
        pts_curr, 0.0, N_T0_STEPS, alpha_iso=ALPHA_ISO,
        sigma_move=0.02, seed=rep_seed + 999,
        sample_at=T0_SAMPLE_SCHEDULE, F_init=F_curr
    )
    pts_curr = result_T0['pts_final']
    F_curr = result_T0['F_final']
    kappa_samples_T0 = result_T0['kappa_samples']
    sample_steps_T0 = result_T0['sample_steps']
    F_at_T0_samples = result_T0['F_at_samples']

    # Final state
    _, _, _, kappa_final, iso_final = compute_F_vec(pts_curr, max(ALPHA_ISO, 1.0))

    elapsed_replica = time.time() - t_replica_start
    print(f'\n  ✓ Réplica {rep_idx} completada en {elapsed_replica:.0f}s ({elapsed_replica/60:.1f} min)')
    print(f'    F final = {F_curr:.2f}, var(κ) = {kappa_final.var():.5f}, ⟨I⟩ = {iso_final.mean():.4f}')

    # Guardar
    np.savez(fname,
             pts_final=pts_curr,
             kappa_final=kappa_final,
             iso_final=iso_final,
             F_final=F_curr,
             kappa_samples_pre=kappa_samples_pre,
             sample_steps_pre=sample_steps_pre,
             kappa_samples_T0=kappa_samples_T0,
             sample_steps_T0=sample_steps_T0,
             F_at_T0_samples=F_at_T0_samples,
             total_time=elapsed_replica)
    print(f'    guardada en {fname}')

    all_aging[rep_idx] = {
        'pts_final': pts_curr,
        'kappa_final': kappa_final,
        'F_final': F_curr,
        'kappa_samples_pre': kappa_samples_pre,
        'sample_steps_pre': sample_steps_pre,
        'kappa_samples_T0': kappa_samples_T0,
        'sample_steps_T0': sample_steps_T0,
        'F_at_T0_samples': F_at_T0_samples,
        'seed': rep_seed,
        'time': elapsed_replica
    }

elapsed_total = time.time() - t_global_start
print(f'\n{"="*70}')
print(f'TODAS LAS RÉPLICAS DE AGING COMPLETADAS')
print(f'Tiempo total: {elapsed_total/60:.1f} min ({elapsed_total/3600:.2f} h)')
print(f'{"="*70}')

---
## ANÁLISIS — Tres tests con criterios pre-registrados

| Test | Criterio | Pre-registrado |
|---|---|---|
| 1. Aging | τ_relax(t_w=25k) / τ_relax(t_w=1k) | > 2.0 |
| 2. χ_4 | peak χ_4 con N=500 | > 5 |
| 3. Sweep N | χ_4_max(2N) / χ_4_max(N) | > 1.5 |

Los criterios fueron definidos en la conversación previa (post-EA, antes de correr este experimento).

In [ ]:
# Resumen por réplica
print('Resumen por réplica:\n')
print(f'{"Rep":>4} {"seed":>5} {"F_final":>10} {"var(κ)":>10} {"⟨I":>8}')
print('-'*50)
for ridx in sorted(all_aging.keys()):
    r = all_aging[ridx]
    _, _, _, _, iso_f = compute_F_vec(r['pts_final'], max(ALPHA_ISO, 1.0))
    print(f'{ridx:>4} {r["seed"]:>5} {r["F_final"]:>10.2f} '
          f'{r["kappa_final"].var():>10.5f} {iso_f.mean():>8.4f}')

# Comparación rápida con experimento EA previo
F_aging = np.array([all_aging[i]['F_final'] for i in sorted(all_aging.keys())])
print(f'\nF finales aging: media={F_aging.mean():.2f}, std={F_aging.std():.2f}, CV={F_aging.std()/F_aging.mean():.3f}')
print(f'  rango: [{F_aging.min():.2f}, {F_aging.max():.2f}]')
print(f'  (experimento EA previo: media=12.96, std=1.50, CV=0.116)')

In [ ]:
# ============================================================
# TEST 1 — AGING (rotura de TTI)
# ============================================================
# Para cada t_w en {1k, 5k, 25k} pasos a T=0:
#   C(t_w, t_w + τ) = correlación nodo-a-nodo entre snapshots t_w y t_w+τ
# Promediar sobre réplicas, fittear stretched exp, comparar τ_relax(t_w).
#
# Criterio pre-registrado: τ(25k)/τ(1k) > 2.0 → AGING confirmado.

print('='*72)
print('TEST 1 — AGING: rotura de Time Translation Invariance')
print('='*72)

# Recopilar samples de T=0 de todas las réplicas
samples_T0 = np.stack([all_aging[r]['kappa_samples_T0']
                        for r in sorted(all_aging.keys())])  # (R, T_snaps, N)
steps_T0 = all_aging[0]['sample_steps_T0']
R_aging, T_snaps, N_sites = samples_T0.shape
print(f'\nForma de samples_T0: {samples_T0.shape}  (R réplicas, T snapshots, N sitios)')
print(f'Pasos de muestreo en T=0: primeros={steps_T0[:5]}, últimos={steps_T0[-5:]}')

def C_ij_per_replica(samples, i, j):
    """Correlación espacial centrada entre snapshots i y j, por réplica."""
    R = samples.shape[0]
    out = np.zeros(R)
    for r in range(R):
        a = samples[r, i] - samples[r, i].mean()
        b = samples[r, j] - samples[r, j].mean()
        denom = np.sqrt(np.mean(a**2) * np.mean(b**2))
        out[r] = np.mean(a*b) / denom if denom > 1e-12 else 0
    return out

# Anclas t_w: snapshots más cercanos a {1000, 5000, 25000}
T_W_TARGETS = [1000, 5000, 25000]
tw_indices = []
for tw in T_W_TARGETS:
    idx = int(np.argmin(np.abs(steps_T0 - tw)))
    tw_indices.append(idx)
    print(f'  t_w target {tw}, snap más cercano: idx={idx}, paso={steps_T0[idx]}')

# Calcular C(t_w, t_w+τ) para cada t_w
def stretched(t, tau, beta):
    return np.exp(-(t/tau)**beta)

aging_results = {}
print('\n--- Fits stretched para cada t_w ---')
for tw_target, tw_idx in zip(T_W_TARGETS, tw_indices):
    tw_actual = steps_T0[tw_idx]
    future_idx = list(range(tw_idx, len(steps_T0)))
    taus = steps_T0[future_idx] - tw_actual

    Cs_per_rep = np.zeros((R_aging, len(future_idx)))
    for k, j in enumerate(future_idx):
        Cs_per_rep[:, k] = C_ij_per_replica(samples_T0, tw_idx, j)

    C_mean = Cs_per_rep.mean(axis=0)
    C_err = Cs_per_rep.std(axis=0, ddof=1) / np.sqrt(R_aging)

    # Fit stretched exp (excluir τ=0 que es trivial; mantener C > 0.05)
    mask = (taus > 0) & (C_mean > 0.05)
    if mask.sum() >= 4:
        try:
            p, cov = curve_fit(stretched, taus[mask].astype(float), C_mean[mask],
                               p0=[1000, 0.5], bounds=([10, 0.1], [1e6, 2.0]),
                               maxfev=10000)
            errs = np.sqrt(np.diag(cov))
            tau_relax, beta_relax = p[0], p[1]
            print(f'  t_w={tw_actual:>6}  τ={tau_relax:>7.0f} ± {errs[0]:.0f}  '
                  f'β={beta_relax:.3f} ± {errs[1]:.3f}  (n_pts={mask.sum()})')
        except Exception as e:
            tau_relax, beta_relax = np.nan, np.nan
            errs = [np.nan, np.nan]
            print(f'  t_w={tw_actual}: fit failed ({e})')
    else:
        tau_relax, beta_relax = np.nan, np.nan
        errs = [np.nan, np.nan]
        print(f'  t_w={tw_actual}: insuficientes puntos')

    aging_results[tw_target] = {
        'tw_actual': tw_actual, 'taus': taus, 'C_mean': C_mean, 'C_err': C_err,
        'tau_relax': tau_relax, 'beta_relax': beta_relax,
        'tau_err': errs[0], 'beta_err': errs[1]
    }

# Veredicto Test 1
tau_low = aging_results[T_W_TARGETS[0]]['tau_relax']
tau_high = aging_results[T_W_TARGETS[-1]]['tau_relax']
tau_ratio = tau_high / tau_low if (np.isfinite(tau_low) and np.isfinite(tau_high) and tau_low > 0) else np.nan

print(f'\n--- VEREDICTO TEST 1 ---')
print(f'τ(t_w={T_W_TARGETS[0]}) = {tau_low:.0f}')
print(f'τ(t_w={T_W_TARGETS[-1]}) = {tau_high:.0f}')
print(f'Ratio τ(25k)/τ(1k) = {tau_ratio:.2f}')
print(f'Criterio pre-registrado: > 2.0')
if np.isfinite(tau_ratio):
    if tau_ratio > 2.0:
        print(f'✓ AGING CONFIRMADO (firma dinámica de vidrio)')
        TEST1_PASS = True
    elif tau_ratio > 1.2:
        print(f'~ AGING MARGINAL')
        TEST1_PASS = False
    else:
        print(f'✗ TTI VALE — no es aging')
        TEST1_PASS = False
else:
    print(f'? Test no concluyente (fits fallaron)')
    TEST1_PASS = False

In [ ]:
# Plot Test 1 — colapso de C(τ) o aging
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: C(τ) para cada t_w
ax = axes[0]
colors = {1000: 'royalblue', 5000: 'darkgreen', 25000: 'darkred'}
for tw_target in T_W_TARGETS:
    res = aging_results[tw_target]
    if np.isfinite(res['tau_relax']):
        # Datos
        ax.errorbar(res['taus'], res['C_mean'], yerr=res['C_err'],
                    fmt='o', color=colors[tw_target], markersize=6,
                    label=f't_w = {res["tw_actual"]} (τ={res["tau_relax"]:.0f}, β={res["beta_relax"]:.2f})')
        # Fit
        t_smooth = np.logspace(1, np.log10(max(res['taus'])+1), 200)
        ax.plot(t_smooth, stretched(t_smooth, res['tau_relax'], res['beta_relax']),
                '-', color=colors[tw_target], alpha=0.6, lw=1.5)

ax.set_xlabel('τ = t - t_w  (paso MC)', fontsize=11)
ax.set_ylabel('C(t_w, t_w + τ)', fontsize=11)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_ylim(0.02, 1.2)
ax.set_title('Test 1: rotura de TTI\n(curvas separadas → aging)', fontsize=11)
ax.legend(fontsize=9, loc='lower left')
ax.grid(alpha=0.3, which='both')

# Panel derecho: τ_relax(t_w) en escala lin-log
ax = axes[1]
tw_vals = [aging_results[tw]['tw_actual'] for tw in T_W_TARGETS
           if np.isfinite(aging_results[tw]['tau_relax'])]
tau_vals = [aging_results[tw]['tau_relax'] for tw in T_W_TARGETS
            if np.isfinite(aging_results[tw]['tau_relax'])]
tau_errs = [aging_results[tw]['tau_err'] for tw in T_W_TARGETS
            if np.isfinite(aging_results[tw]['tau_relax'])]

ax.errorbar(tw_vals, tau_vals, yerr=tau_errs, fmt='o-', markersize=10, lw=2, color='crimson')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('t_w (waiting time, paso MC)', fontsize=11)
ax.set_ylabel('τ_relax (paso MC)', fontsize=11)
ax.set_title(f'τ_relax(t_w): ratio={tau_ratio:.2f}× (criterio > 2)', fontsize=11)
ax.axhline(tau_vals[0]*2 if len(tau_vals)>0 else 0, ls='--', color='black', alpha=0.5,
           label='2× τ(1k) (umbral aging)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

plt.suptitle(f'TEST 1 — AGING  ({("PASA" if TEST1_PASS else "NO PASA")})',
             fontsize=13, y=1.02)
plt.tight_layout()
save_plot('200_aging_test1')
plt.show()

In [ ]:
# ============================================================
# TEST 2 — χ_4(t) susceptibilidad de cuatro puntos
# ============================================================
# Para cada réplica r y snapshot t:
#   q_r(t) = (1/N) Σ_i δκ_r(t)·δκ_r(0) / σ²(0)
# χ_4(t) = N · Var_r [ q_r(t) ]
# Pico → cooperatividad ξ_4 ~ √χ_4_peak
#
# Criterio pre-registrado: peak χ_4 > 5 con N=500.

print('='*72)
print('TEST 2 — χ_4(t) susceptibilidad de cuatro puntos')
print('='*72)

# q_r(t=0, t=k) por réplica
q_per_rep = np.zeros((R_aging, T_snaps))
ref_idx = 0  # snapshot al inicio de T=0
for r in range(R_aging):
    ref = samples_T0[r, ref_idx] - samples_T0[r, ref_idx].mean()
    var_ref = np.mean(ref**2)
    for k in range(T_snaps):
        cur = samples_T0[r, k] - samples_T0[r, k].mean()
        q_per_rep[r, k] = np.mean(ref*cur) / var_ref if var_ref > 1e-12 else 0

chi4 = N_sites * np.var(q_per_rep, axis=0, ddof=1)

# Bootstrap CI 95% para χ_4
print(f'\nCalculando bootstrap CI 95% (n=2000)...')
rng_boot = np.random.default_rng(99)
N_BOOT = 2000
chi4_boot = np.zeros((N_BOOT, T_snaps))
for b in range(N_BOOT):
    idx = rng_boot.choice(R_aging, R_aging, replace=True)
    chi4_boot[b] = N_sites * np.var(q_per_rep[idx], axis=0, ddof=1)
chi4_lo = np.percentile(chi4_boot, 2.5, axis=0)
chi4_hi = np.percentile(chi4_boot, 97.5, axis=0)

# Pico
peak_idx = int(np.argmax(chi4))
peak_t = steps_T0[peak_idx]
peak_val = chi4[peak_idx]
peak_lo = chi4_lo[peak_idx]
peak_hi = chi4_hi[peak_idx]

print(f'\n--- χ_4(t) ---')
print(f'  Pico: χ_4_max = {peak_val:.2f} (CI 95%: [{peak_lo:.2f}, {peak_hi:.2f}])')
print(f'  Localización del pico: t = {peak_t} pasos')
print(f'  ξ_4 ≈ √χ_4_max ≈ {np.sqrt(peak_val):.2f} (longitud cooperativa)')

print(f'\n--- VEREDICTO TEST 2 ---')
print(f'Criterio pre-registrado: peak χ_4 > 5 con N=500')
if peak_val > 5:
    print(f'✓ COOPERATIVIDAD CONFIRMADA (χ_4 = {peak_val:.2f} > 5)')
    TEST2_PASS = True
elif peak_val > 2:
    print(f'~ COOPERATIVIDAD DÉBIL (χ_4 = {peak_val:.2f} ∈ [2,5])')
    TEST2_PASS = False
else:
    print(f'✗ NO COOPERATIVO (χ_4 = {peak_val:.2f} < 2)')
    TEST2_PASS = False

In [ ]:
# Plot Test 2 — χ_4(t) con bootstrap CI
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: χ_4(t)
ax = axes[0]
ax.fill_between(steps_T0, chi4_lo, chi4_hi, alpha=0.3, color='steelblue',
                label='IC 95% bootstrap')
ax.plot(steps_T0, chi4, 'o-', color='steelblue', lw=2, markersize=6, label='χ_4(t)')
ax.axvline(peak_t, color='red', ls='--', alpha=0.7,
           label=f'pico en t={peak_t}')
ax.axhline(5.0, color='black', ls=':', alpha=0.7, label='umbral pre-registrado (=5)')
ax.set_xlabel('t (paso MC desde quench)', fontsize=11)
ax.set_ylabel('χ_4(t)', fontsize=11)
ax.set_xscale('log')
ax.set_title(f'χ_4(t): peak = {peak_val:.2f} en t={peak_t}', fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')

# Panel derecho: ⟨q(t)⟩ y Var[q(t)] separados
ax = axes[1]
q_mean = q_per_rep.mean(axis=0)
q_var = q_per_rep.var(axis=0, ddof=1)
ax_twin = ax.twinx()
l1 = ax.plot(steps_T0, q_mean, 'o-', color='royalblue', lw=2, markersize=6, label='⟨q(t)⟩')
l2 = ax_twin.plot(steps_T0, q_var, 's-', color='crimson', lw=2, markersize=5, label='Var[q(t)]')
ax.set_xlabel('t (paso MC desde quench)', fontsize=11)
ax.set_ylabel('⟨q(t)⟩', fontsize=11, color='royalblue')
ax_twin.set_ylabel('Var[q(t)]', fontsize=11, color='crimson')
ax.set_xscale('log')
ax.tick_params(axis='y', labelcolor='royalblue')
ax_twin.tick_params(axis='y', labelcolor='crimson')
ax.set_title('⟨q⟩ decae mientras Var[q] tiene pico\n(=fluctuaciones cooperativas)', fontsize=11)
lns = l1 + l2
ax.legend(lns, [l.get_label() for l in lns], fontsize=9, loc='center right')
ax.grid(alpha=0.3, which='both')

plt.suptitle(f'TEST 2 — χ_4 dinámica  ({("PASA" if TEST2_PASS else "NO PASA")})',
             fontsize=13, y=1.02)
plt.tight_layout()
save_plot('201_chi4_test2')
plt.show()

---
## TEST 3 — Sweep de tamaño N (OPCIONAL)

**Costo:** ~14 horas adicionales (3 valores de N × 5 réplicas × t_max=30000).

**Recomendación:** activar `RUN_SIZE_SCALING = True` solo si Tests 1 ó 2 dieron positivo. En transición de vidrio termodinámica, χ_4_max diverge con N. Si satura, es crossover (no transición real).

**Criterio pre-registrado:** χ_4_max(N=2000) / χ_4_max(N=500) > 1.5

In [ ]:
# TEST 3 — Sweep N (solo si RUN_SIZE_SCALING)
# Estructura: por cada N en [500, 1000, 2000], 5 réplicas con protocolo aging.
# Reusa todas las funciones; cambia solo N_TARGET interno.

if not RUN_SIZE_SCALING:
    print('Test 3 desactivado (RUN_SIZE_SCALING = False).')
    print('Para activarlo: setear RUN_SIZE_SCALING = True en celda de configuración.')
    print('Costo estimado: +14 horas.\n')
    TEST3_PASS = None
    size_results = None
else:
    print('='*72)
    print('TEST 3 — SWEEP DE TAMAÑO N')
    print('='*72)

    SIZE_DIR = 'data_size'
    os.makedirs(SIZE_DIR, exist_ok=True)
    N_VALUES = [500, 1000, 2000]
    SIZE_REPS = 5
    N_T0_SIZE = 30000  # menos pasos para test 3 (más caro)
    SIZE_SCHEDULE_T0 = build_t0_schedule(N_T0_SIZE)

    size_replicas = {}

    for N_size in N_VALUES:
        print(f'\n{"="*70}\nSweep N = {N_size}\n{"="*70}')

        # Config inicial específica de este N
        pts_init_size, _ = init_uniform_T3(N_size, seed=42, L=L_T3)
        F_init_size, _, _, _, _ = compute_F_vec(pts_init_size, ALPHA_ISO)

        # Calibrar α_iso para este N (por consistencia)
        _, fv, fi, _, _ = compute_F_vec(pts_init_size, alpha_iso=1.0)
        alpha_size = fv / max(fi, 1e-6)
        print(f'  α_iso({N_size}) = {alpha_size:.4f}')

        size_replicas[N_size] = []
        size_seeds_base = 5000 + N_size

        for r_idx in range(SIZE_REPS):
            seed = size_seeds_base + r_idx
            fname = os.path.join(SIZE_DIR, f'size_N{N_size}_rep{r_idx}_seed{seed}.npz')
            if os.path.exists(fname):
                d = np.load(fname, allow_pickle=True)
                size_replicas[N_size].append({
                    'kappa_samples_T0': d['kappa_samples_T0'],
                    'sample_steps_T0': d['sample_steps_T0']
                })
                print(f'  Rep {r_idx}: cargada ({fname})')
                continue

            print(f'\n  Réplica {r_idx} (seed {seed})')
            t_start = time.time()
            pts_curr = pts_init_size.copy()
            F_curr = F_init_size

            # Pre-annealing
            for ph_idx, (Tp, ns) in enumerate(zip(T_SCHEDULE_PRE, N_STEPS_PRE)):
                res = run_mc_phase(pts_curr, Tp, ns, alpha_iso=alpha_size,
                                   sigma_move=0.02, seed=seed+ph_idx,
                                   sample_at=[], F_init=F_curr, verbose=True)
                pts_curr = res['pts_final']
                F_curr = res['F_final']

            # T=0 con sampling
            res_T0 = run_mc_phase(pts_curr, 0.0, N_T0_SIZE, alpha_iso=alpha_size,
                                  sigma_move=0.02, seed=seed+999,
                                  sample_at=SIZE_SCHEDULE_T0, F_init=F_curr,
                                  verbose=True)

            np.savez(fname,
                     kappa_samples_T0=res_T0['kappa_samples'],
                     sample_steps_T0=res_T0['sample_steps'])
            size_replicas[N_size].append({
                'kappa_samples_T0': res_T0['kappa_samples'],
                'sample_steps_T0': res_T0['sample_steps']
            })
            print(f'    completada en {(time.time()-t_start)/60:.1f} min')

    # Análisis: χ_4_max para cada N
    size_results = {}
    print(f'\n--- χ_4 por tamaño N ---')
    for N_size in N_VALUES:
        reps = size_replicas[N_size]
        samp = np.stack([r['kappa_samples_T0'] for r in reps])
        steps = reps[0]['sample_steps_T0']
        Rn, Tn, Nn = samp.shape
        q = np.zeros((Rn, Tn))
        for r in range(Rn):
            ref = samp[r, 0] - samp[r, 0].mean()
            vr = np.mean(ref**2)
            for k in range(Tn):
                cur = samp[r, k] - samp[r, k].mean()
                q[r, k] = np.mean(ref*cur)/vr if vr > 1e-12 else 0
        chi4_n = Nn * np.var(q, axis=0, ddof=1)
        peak = float(chi4_n.max())
        size_results[N_size] = {'chi4': chi4_n, 'steps': steps, 'peak': peak}
        print(f'  N={N_size:>5}: peak χ_4 = {peak:.2f}')

    # Veredicto Test 3
    ratio_size = size_results[2000]['peak'] / size_results[500]['peak']
    print(f'\n--- VEREDICTO TEST 3 ---')
    print(f'χ_4_max(N=2000) / χ_4_max(N=500) = {ratio_size:.2f}')
    print(f'Criterio pre-registrado: > 1.5')
    if ratio_size > 1.5:
        print(f'✓ ξ_4 CRECIENTE con N (vidrio termodinámico)')
        TEST3_PASS = True
    else:
        print(f'✗ ξ_4 SATURA (crossover finito, no transición termodinámica)')
        TEST3_PASS = False

In [ ]:
# Plot Test 3 (solo si RUN_SIZE_SCALING)
if RUN_SIZE_SCALING and size_results is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # χ_4(t) por N
    ax = axes[0]
    cmap_n = {500: 'royalblue', 1000: 'darkgreen', 2000: 'darkred'}
    for N_size in N_VALUES:
        sr = size_results[N_size]
        ax.plot(sr['steps'], sr['chi4'], 'o-', color=cmap_n[N_size],
                lw=2, markersize=6, label=f'N={N_size}, peak={sr["peak"]:.2f}')
    ax.set_xlabel('t (paso MC)', fontsize=11)
    ax.set_ylabel('χ_4(t)', fontsize=11)
    ax.set_xscale('log')
    ax.set_title('χ_4(t) para distintos N', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3, which='both')

    # Escaling χ_4_max(N)
    ax = axes[1]
    Ns = np.array(N_VALUES)
    peaks = np.array([size_results[n]['peak'] for n in Ns])
    ax.loglog(Ns, peaks, 'o-', markersize=12, lw=2, color='crimson')
    if len(Ns) >= 2:
        gamma, logC = np.polyfit(np.log(Ns), np.log(peaks), 1)
        Ns_smooth = np.logspace(np.log10(Ns.min()), np.log10(Ns.max()), 100)
        ax.plot(Ns_smooth, np.exp(logC) * Ns_smooth**gamma,
                '--', color='black', alpha=0.6, label=f'ajuste: χ_4 ∝ N^{gamma:.2f}')
    ax.set_xlabel('N', fontsize=11)
    ax.set_ylabel('χ_4_max', fontsize=11)
    ax.set_title(f'Escaling χ_4_max(N): ratio={ratio_size:.2f}× (criterio > 1.5)', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3, which='both')

    plt.suptitle(f'TEST 3 — sweep N  ({("PASA" if TEST3_PASS else "NO PASA")})',
                 fontsize=13, y=1.02)
    plt.tight_layout()
    save_plot('202_size_test3')
    plt.show()
else:
    print('Plot Test 3 omitido (sweep N desactivado)')

In [ ]:
# ============================================================
# VEREDICTO FINAL — combinación de los 3 tests
# ============================================================
print('='*72)
print('VEREDICTO FINAL — fase de vidrio del sustrato DEE')
print('='*72)

VERDICT_TABLE = {
    (True, True, True):    ('VIDRIO ESTRUCTURAL CONFIRMADO',
                            '3/3: aging + cooperatividad + escala finita coherente. '
                            'Resultado limpio para incorporación a §6.18quinquies.'),
    (True, True, False):   ('vidrio dinámico finite-size',
                            '2/3: aging + χ_4 OK pero ξ_4 satura. '
                            'Es vidrio en N≤500, pero no transición termodinámica.'),
    (True, False, True):   ('AGING + escala — anómalo, revisar protocolo',
                            'Aging y size effect SIN cooperatividad: investigar bug.'),
    (True, False, False):  ('aging modesto sin cooperatividad',
                            '1/3: relajación lenta no exponencial pero sin cluster cooperativo. '
                            'Más bien líquido sobre-enfriado lento.'),
    (False, True, True):   ('cooperatividad + escala sin aging — improbable',
                            'TTI vale pero hay χ_4 grande: investigar artefacto.'),
    (False, True, False):  ('cooperatividad sin aging ni escala',
                            'Heterogeneidad dinámica transitoria. No es vidrio.'),
    (False, False, True):  ('size effects sin firma dinámica',
                            'Probable artefacto de muestreo / inicialización.'),
    (False, False, False): ('NO ES VIDRIO',
                            '0/3: cristal con defectos congelados o líquido fragil ergódico. '
                            'Reportar negativo y ajustar interpretación de defectos blandos.'),
}

print(f'\nTest 1 (Aging):     {"✓ PASA" if TEST1_PASS else "✗ NO PASA"}  '
      f'(τ ratio = {tau_ratio:.2f}, umbral 2.0)')
print(f'Test 2 (χ_4):       {"✓ PASA" if TEST2_PASS else "✗ NO PASA"}  '
      f'(peak χ_4 = {peak_val:.2f}, umbral 5.0)')
if TEST3_PASS is None:
    print(f'Test 3 (sweep N):   — desactivado —')
    n_pass = int(TEST1_PASS) + int(TEST2_PASS)
    print(f'\nCriterios pasados: {n_pass}/2 (test 3 no corrido)')

    print(f'\n--- VEREDICTO PARCIAL ---')
    if TEST1_PASS and TEST2_PASS:
        print('✓ Tests dinámicos POSITIVOS (aging + cooperatividad).')
        print('  Recomendación: activar RUN_SIZE_SCALING y correr Test 3 para confirmación termodinámica.')
    elif TEST1_PASS or TEST2_PASS:
        print(f'~ Resultado parcial ({n_pass}/2).')
        print('  Recomendación: documentar como vidrio dinámico marginal en §6.18quinquies,')
        print('                  sin pretender confirmación termodinámica.')
    else:
        print('✗ Tests dinámicos NEGATIVOS.')
        print('  Recomendación: documentar como cristal con defectos / líquido lento honestamente.')
else:
    key = (TEST1_PASS, TEST2_PASS, TEST3_PASS)
    name, descr = VERDICT_TABLE[key]
    n_pass = sum(key)
    print(f'Test 3 (sweep N):   {"✓ PASA" if TEST3_PASS else "✗ NO PASA"}  '
          f'(ratio = {ratio_size:.2f}, umbral 1.5)')
    print(f'\nCriterios pasados: {n_pass}/3')
    print(f'\n--- VEREDICTO ---')
    print(f'  {name}')
    print(f'  {descr}')

# Guardar veredicto en JSON
verdict_summary = {
    'test1_aging': {'pass': bool(TEST1_PASS), 'tau_ratio': float(tau_ratio) if np.isfinite(tau_ratio) else None,
                    'criterion': 'tau(25k)/tau(1k) > 2.0',
                    'taus': {str(tw): float(aging_results[tw]['tau_relax']) for tw in T_W_TARGETS
                             if np.isfinite(aging_results[tw]['tau_relax'])},
                    'betas': {str(tw): float(aging_results[tw]['beta_relax']) for tw in T_W_TARGETS
                              if np.isfinite(aging_results[tw]['beta_relax'])}},
    'test2_chi4':  {'pass': bool(TEST2_PASS), 'peak_chi4': float(peak_val),
                    'peak_t': int(peak_t),
                    'ci95': [float(peak_lo), float(peak_hi)],
                    'criterion': 'peak chi_4 > 5'},
    'test3_size':  {'pass': bool(TEST3_PASS) if TEST3_PASS is not None else None,
                    'ratio': float(ratio_size) if (TEST3_PASS is not None) else None,
                    'criterion': 'chi_4(2N)/chi_4(N) > 1.5',
                    'run': bool(RUN_SIZE_SCALING)},
    'n_replicas': N_REPLICAS,
    'N_sites': N_TARGET,
    'N_T0_steps': N_T0_STEPS
}
with open(os.path.join(DATA_DIR, 'verdict_summary.json'), 'w') as f:
    json.dump(verdict_summary, f, indent=2)
print(f'\nResumen JSON guardado en {DATA_DIR}/verdict_summary.json')

In [ ]:
# Empaquetar resultados para descarga
import shutil
shutil.make_archive('plots_aging', 'zip', PLOT_DIR)
shutil.make_archive('data_aging', 'zip', DATA_DIR)
if RUN_SIZE_SCALING and os.path.exists('data_size'):
    shutil.make_archive('data_size', 'zip', 'data_size')

try:
    from google.colab import files
    files.download('plots_aging.zip')
    files.download('data_aging.zip')
    if RUN_SIZE_SCALING and os.path.exists('data_size.zip'):
        files.download('data_size.zip')
except ImportError:
    pass

print('Listo.')
print('Plots en plots_aging.zip')
print('Datos crudos en data_aging.zip')
if RUN_SIZE_SCALING:
    print('Datos sweep N en data_size.zip')
print(f'Tiempo total de la sesión: {(time.time() - t_global_start)/60:.1f} min')